# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Fetch metadata
meta_dict = dataset.metadata.to_json()
print(f"{meta_dict['name']}: {meta_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps understand the structure before extracting data.


In [ ]:
# List all available record sets and field IDs

print("Available record sets in the dataset:\n")
for rs in dataset.metadata.record_sets:
    print(f"- Record Set Name: {getattr(rs, 'name', '<no name>')} | @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name} | @id: {field.id} | dataType: {field.data_type}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Note:** This demo loads the first record set found, but you can adjust the code to select other record sets by their `@id` as shown above.

In [ ]:
# Extract data from record sets using their @id

# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Examine the columns of the first record set dataframe
first_rs_id = record_set_ids[0]
print(f"Loaded DataFrame for record set {first_rs_id}:")
print(dataframes[first_rs_id].columns.tolist())
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numeric fields (e.g., protein abundance or MW), normalizing numeric fields, and grouping by categorical columns. All references should use field `@id` as shown previously.


In [ ]:
# EDA on the first record set DataFrame
df = dataframes[first_rs_id]
# Inspect columns to find a numeric field
print("Columns in the dataframe:")
print(df.columns.tolist())

# Example: Let's pick 'cr:MW' (Molecular Weight) for analysis, adjust if actual columns differ
# If there is a column with MW or abundance, use that; otherwise, print all numeric columns
import numpy as np
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns detected:", numeric_cols)

# If 'cr:MW' exists, use it; else use first numeric column
if 'cr:MW' in df.columns:
    numeric_field_id = 'cr:MW'
elif len(numeric_cols) > 0:
    numeric_field_id = numeric_cols[0]
else:
    numeric_field_id = None

if numeric_field_id:
    threshold = df[numeric_field_id].quantile(0.75)  # filter on top quartile as example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical column, such as 'cr:accession' or 'cr:description'
    group_field = None
    for col in ['cr:accession', 'cr:description', 'cr:Sample']:
        if col in df.columns:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

We'll plot a histogram of the selected numeric field. If a group field exists, we can plot mean values by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and len(df) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        # Top 10 groups by mean value
        grouped = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)[:10]
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped.values, y=grouped.index)
        plt.title(f"Top 10 {group_field} by avg {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.show()
else:
    print("Could not visualize as no numeric field detected.")

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to:

- Load and inspect a dataset described with Croissant schema
- Discover record sets, their fields, and data types
- Load data into pandas DataFrames
- Perform basic exploratory analysis, filtering, normalization, and grouping
- Visualize results based on field `@id`

You are encouraged to further explore other record sets, fields, and perform domain-specific analysis tailored to your research needs.
